# Forgetting Ledger — ResNet-18 + BatchNorm on Split CIFAR-10 (Kaggle)

1. *Settings → Accelerator → GPU T4 x2 or P100*, *Internet → On*.
2. Only if the repository is private: *Add-ons → Secrets* → **`GH_TOKEN`** (GitHub token with read access).
3. Kaggle sessions stop after 12 h and only `/kaggle/working` is kept when you **Save Version (Save & Run All)**. To continue in a new session, add the previous version's output as an input dataset: the first cell copies `runs/` back and every job resumes from its checkpoint.

In [ ]:
import os, glob, shutil
RUN_ROOT, DATA_ROOT, REPO_DIR = '/kaggle/working/runs', '/kaggle/tmp/data', '/kaggle/tmp/forgetting-ledger'
os.makedirs(RUN_ROOT, exist_ok=True)
# resume from a previous version's output attached as input
for prev in glob.glob('/kaggle/input/*/runs'):
    print('restoring', prev)
    shutil.copytree(prev, RUN_ROOT, dirs_exist_ok=True)

In [ ]:
import subprocess
from kaggle_secrets import UserSecretsClient
try:
    tok = UserSecretsClient().get_secret('GH_TOKEN')
except Exception:
    tok = None   # public repository: no token needed
url = f'https://{tok}@github.com/Basil-Mohammad/forgetting-ledger.git' if tok else 'https://github.com/Basil-Mohammad/forgetting-ledger.git'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', url, REPO_DIR], check=True)
subprocess.run(['pip', '-q', 'install', '-r', f'{REPO_DIR}/requirements.txt'], check=True)
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# Smoke test on this GPU (about 5 minutes). Run once before the real jobs.
import subprocess, sys, time
t0 = time.time()
# the whole pipeline (train, scores, removal, surgery, LDS, freeze) at a tiny scale, so that any problem
# shows up here and not after an hour of real work
subprocess.run([sys.executable, "scripts/run.py", "all", "configs/c10_resnet_bn.yaml", "seed=99",
                "train_per_task=320", "first_task_train=640", "train.first_task_epochs=1", "train.epochs=1",
                "probe_per_class=4", "ledger.tracin_checkpoints=3", "interventions.reps=1",
                "interventions.removal_fracs=[0.1]", "interventions.lds_subsets=2", "interventions.surgery_targets=1",
                "interventions.param_fracs=[0.1]", f"out_root={RUN_ROOT}/../_smoke", f"data_root={DATA_ROOT}"],
               cwd=REPO_DIR, check=True)
print(f"smoke test OK in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# One line per job: <command> <config> [overrides...]
# `all` = train -> scores -> removal -> surgery -> lds -> params (every step resumable).
# Phase 1 (paper, Section "Scale and batch normalisation"): reduced ResNet-18 with BatchNorm on
# Split CIFAR-10, ten seeds. About 1 h per seed on a T4.
JOBS = "\n".join(f"all configs/c10_resnet_bn.yaml seed={s}" for s in range(10))
JOBS = [l.split() for l in JOBS.strip().splitlines() if l.strip() and not l.startswith("#")]
print(len(JOBS), "jobs")

In [ ]:
# Kaggle hard limit is 12 h: stop starting new jobs after 11 h so the version can be saved cleanly.
import time; T_START = time.time(); LIMIT_H = 11.0
import subprocess, sys, time, os
def run_job(job):
    cmd = [sys.executable, "scripts/run.py", *job, f"out_root={RUN_ROOT}", f"data_root={DATA_ROOT}"]
    print(">>>", " ".join(job), flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"<<< exit {p.returncode} after {(time.time()-t0)/60:.1f} min", flush=True)
    return p.returncode

# Re-running this cell after a disconnect resumes exactly where it stopped:
# finished steps are skipped and the current step restarts from state/latest.pt.
for job in JOBS:
    if (time.time() - T_START) / 3600 > LIMIT_H:
        print('time budget reached - Save Version and continue in a new session'); break
    if run_job(job) != 0:
        print("job failed - fix and re-run this cell (it will resume)"); break

In [ ]:
# Progress overview: which seeds are finished, forgetting and completeness of the ledger.
import json, glob, os
for d in sorted(glob.glob(f"{RUN_ROOT}/scifar10-task-resnet18r-bn-finetune-s*")):
    m = [json.loads(l) for l in open(f"{d}/metrics.jsonl")] if os.path.exists(f"{d}/metrics.jsonl") else []
    steps = sorted(os.listdir(f"{d}/interventions")) if os.path.isdir(f"{d}/interventions") else []
    t1 = [x for x in m if x.get("task") == 1]
    if t1:
        acc0 = [x for x in m if x.get("task") == 0][0]["task_acc"][0]
        print(os.path.basename(d), f"forgetting {100 * (acc0 - t1[0]['task_acc'][0]):.1f} pp",
              f"completeness eps {100 * t1[0]['completeness']:.2f} %", "interventions:", steps)
    else:
        print(os.path.basename(d), "task B not finished yet")

In [ ]:
# Compact archive for the paper analysis (ledgers, scores, interventions, evaluations; model snapshots
# and resumable states are excluded). Download flgr_results.zip and send it back.
import os, subprocess
subprocess.run(f"cd {RUN_ROOT} && zip -qr ../flgr_results.zip . -x '*/snapshots/*' '*/state/*'", shell=True, check=True)
print(f"{RUN_ROOT}/../flgr_results.zip", round(os.path.getsize(f"{RUN_ROOT}/../flgr_results.zip") / 1e6, 1), "MB")